# 🚗 RoadVision AI — Final Pothole Detector Training Pipeline
**Dataset:** RDD2022 India (D40 Potholes) + Local Smartphone Dataset V1 (1,598 Images | 3,675 Bounding Boxes)
**Baseline Checkpoint:** `pothole_best.pt` (Pretrained Road Hazard YOLOv8s)
**Hardware Target:** Google Colab NVIDIA Tesla T4 GPU

---
### 📌 Workflow Overview:
1. **GPU Acceleration Check** (`nvidia-smi`)
2. **Install Ultralytics YOLOv8**
3. **Upload & Pre-Extraction Integrity Verification** (Supports Google Drive OR Direct Upload with CRC-32 + Byte-Size verification)
4. **Safe Extraction & Complete YOLO Dataset Audit** (Zero temporal leakage verification)
5. **Fine-Tuning Execution** (25 Epochs, imgsz=640, batch=16, AdamW, lr0=0.002)
6. **Checkpoint Verification & Automatic Download as `pothole_rdd_india_best.pt`**

In [ ]:
# =====================================================================
# STEP 1: Verify NVIDIA GPU Acceleration (Tesla T4)
# =====================================================================
import torch

print("Checking CUDA acceleration...")
if not torch.cuda.is_available():
    raise SystemError(
        "❌ ERROR: No GPU detected!\n"
        "Please enable GPU acceleration:\n"
        "  1. Click 'Runtime' in the top menu\n"
        "  2. Click 'Change runtime type'\n"
        "  3. Select 'T4 GPU' under Hardware Accelerator\n"
        "  4. Click 'Save' and re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU Online: {gpu_name} ({vram_gb:.2f} GB VRAM)")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# =====================================================================
# STEP 2: Install Ultralytics YOLOv8 & Dependencies
# =====================================================================
!pip install -q --upgrade ultralytics pyyaml

import ultralytics
print(f"✅ Ultralytics YOLOv8 installed (Version: {ultralytics.__version__})")

In [ ]:
# =====================================================================
# STEP 3: Robust Transfer & Pre-Extraction Integrity Verification
# =====================================================================
# Automatically checks Google Drive first (fastest & 100% reliable),
# or falls back to direct browser upload in /content/.
# Verifies exact byte size & zip CRC-32 to catch browser truncation.
# =====================================================================
import os
import sys
import shutil
import zipfile
import hashlib

EXPECTED_ZIP_NAME = "dataset_rdd_india_potholes.zip"
EXPECTED_ZIP_BYTES = 104640492        # Exact verified local byte size (~99.79 MB)
EXPECTED_ZIP_SHA256 = "8d7f1575b5874f1383a5e74c359aa70b477ade7cd071b12ad098af4d97d8cbeb"
EXPECTED_MODEL_NAME = "pothole_best.pt"
EXPECTED_MODEL_BYTES = 24480547       # Exact verified local byte size (~23.35 MB)

# --- Check 1: Google Drive Transfer (Recommended) ---
drive_zip_candidates = [
    f"/content/drive/MyDrive/{EXPECTED_ZIP_NAME}",
    f"/content/drive/MyDrive/RoadVision/{EXPECTED_ZIP_NAME}",
    f"/content/drive/MyDrive/DL_Project/{EXPECTED_ZIP_NAME}"
]
drive_model_candidates = [
    f"/content/drive/MyDrive/{EXPECTED_MODEL_NAME}",
    f"/content/drive/MyDrive/RoadVision/{EXPECTED_MODEL_NAME}",
    f"/content/drive/MyDrive/DL_Project/{EXPECTED_MODEL_NAME}"
]

# Try mounting drive if folder exists or files not in /content/
target_zip = f"/content/{EXPECTED_ZIP_NAME}"
target_model = f"/content/{EXPECTED_MODEL_NAME}"

if not (os.path.exists(target_zip) and os.path.exists(target_model)):
    print("Files not yet found in /content/. Checking Google Drive...")
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        
        for cand in drive_zip_candidates:
            if os.path.exists(cand):
                print(f"Found dataset in Drive: {cand} -> Copying to /content/...")
                shutil.copy2(cand, target_zip)
                break
                
        for cand in drive_model_candidates:
            if os.path.exists(cand):
                print(f"Found model in Drive: {cand} -> Copying to /content/...")
                shutil.copy2(cand, target_model)
                break
    except Exception as e:
        print(f"Google Drive check skipped or not mounted: {e}")

# --- Check 2: File Existence Verification ---
if not os.path.exists(target_zip):
    raise FileNotFoundError(
        f"❌ ERROR: Missing '{EXPECTED_ZIP_NAME}' in /content/!\n"
        "How to fix:\n"
        "  Option A (Recommended): Place 'dataset_rdd_india_potholes.zip' in your Google Drive (MyDrive) and run this cell.\n"
        "  Option B: Click the Files icon in Colab's left sidebar, click Upload, and select the ZIP file."
    )

if not os.path.exists(target_model):
    raise FileNotFoundError(
        f"❌ ERROR: Missing '{EXPECTED_MODEL_NAME}' in /content/!\n"
        "Please upload 'pothole_best.pt' to /content/ or place it in Google Drive."
    )

# --- Check 3: Exact Byte Size Verification (Catches Browser Truncation) ---
actual_zip_bytes = os.path.getsize(target_zip)
print(f"Verifying dataset archive size: {actual_zip_bytes:,} bytes...")

if actual_zip_bytes != EXPECTED_ZIP_BYTES:
    diff_bytes = EXPECTED_ZIP_BYTES - actual_zip_bytes
    raise ValueError(
        f"❌ CORRUPTED / INCOMPLETE UPLOAD DETECTED!\n"
        f"  Expected size : {EXPECTED_ZIP_BYTES:,} bytes (~99.79 MB)\n"
        f"  Actual uploaded: {actual_zip_bytes:,} bytes (~{actual_zip_bytes/(1024*1024):.2f} MB)\n"
        f"  Missing bytes  : {diff_bytes:,} bytes\n\n"
        "Browser uploads often disconnect silently on large files.\n"
        "RECOMMENDED FIX:\n"
        "  1. Upload 'dataset_rdd_india_potholes.zip' to Google Drive (https://drive.google.com)\n"
        "  2. Mount Google Drive and copy directly (Drive transfers are 100% reliable)."
    )

# --- Check 4: Full CRC-32 & ZIP Structure Verification ---
print("Verifying internal ZIP integrity (CRC-32 checksums)...")
with zipfile.ZipFile(target_zip, 'r') as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise zipfile.BadZipFile(f"❌ ZIP integrity check failed! Corrupt file inside archive: {bad_member}")
    namelist = zf.namelist()
    if 'data.yaml' not in namelist:
        raise ValueError("❌ Archive missing required 'data.yaml' at root!")

print(f"✅ PRE-EXTRACTION AUDIT PASSED! Archive is 100% complete ({len(namelist)} entries, CRC-32 verified).")

In [ ]:
# =====================================================================
# STEP 4: Safe Extraction & Complete Dataset Audit
# =====================================================================
import os
import glob
import zipfile
import yaml
from PIL import Image

DATASET_DIR = "/content/dataset_rdd_india_potholes"

# Clean existing extraction directory if partially extracted
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

print(f"Extracting {target_zip} -> {DATASET_DIR}...")
with zipfile.ZipFile(target_zip, 'r') as zf:
    zf.extractall(DATASET_DIR)

# Reconfigure data.yaml for Google Colab environment
data_yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(data_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = DATASET_DIR
data_cfg["train"] = "images/train"
data_cfg["val"] = "images/val"
data_cfg["names"] = {0: "pothole"}
data_cfg["nc"] = 1

with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

# Run complete audit
train_imgs = glob.glob(f"{DATASET_DIR}/images/train/*")
val_imgs = glob.glob(f"{DATASET_DIR}/images/val/*")
train_lbls = glob.glob(f"{DATASET_DIR}/labels/train/*")
val_lbls = glob.glob(f"{DATASET_DIR}/labels/val/*")

print("\n--- Extracted Dataset Audit ---")
print(f"Train Images : {len(train_imgs):<6} | Train Labels : {len(train_lbls):<6}")
print(f"Val Images   : {len(val_imgs):<6} | Val Labels   : {len(val_lbls):<6}")
print(f"Total Images : {len(train_imgs) + len(val_imgs)}")

# 1:1 image to label parity check
assert len(train_imgs) == len(train_lbls), "❌ Train image and label counts do not match!"
assert len(val_imgs) == len(val_lbls), "❌ Val image and label counts do not match!"

# Zero temporal leakage check (no sequential v1 frames in val)
v1_in_val = [f for f in val_imgs if "v1_" in os.path.basename(f)]
assert len(v1_in_val) == 0, f"❌ LEAKAGE DETECTED: {len(v1_in_val)} local sequential frames found in validation!"

# Zero Rangoon Road contamination check
all_extracted_files = train_imgs + val_imgs
rangoon_leak = [f for f in all_extracted_files if "rangoon" in os.path.basename(f).lower()]
assert len(rangoon_leak) == 0, "❌ CONTAMINATION: Rangoon test frames found in dataset!"

# Verify sample image readability
with Image.open(train_imgs[0]) as img:
    img.verify()

print("\n✅ DATASET VERIFICATION COMPLETE:")
print("  - Parity: 100% 1:1 Image-Label alignment")
print("  - Leakage Policy: Zero local sequential V1 frames in validation")
print("  - Blind Test Isolation: Rangoon Road completely excluded (1,005 frames pure test)")

In [ ]:
# =====================================================================
# STEP 5: Model Weights & Training Hyperparameter Verification
# =====================================================================
from ultralytics import YOLO

print(f"Loading baseline weights: {target_model}...")
model = YOLO(target_model)
print(f"✅ Loaded model architecture: {model.model.__class__.__name__} ({len(model.model.names)} classes)")

# Define locked experimental hyperparameters
TRAIN_CONFIG = {
    "data": data_yaml_path,
    "epochs": 25,
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "optimizer": "AdamW",
    "lr0": 0.002,          # Conservative fine-tuning rate (preserves feature maps)
    "lrf": 0.01,
    "patience": 10,        # Early stopping if mAP plateaus
    "project": "runs/detect",
    "name": "pothole_rdd_india",
    "save": True,
    "augment": True,
    "close_mosaic": 10
}

print("\n--- Training Configuration Locked ---")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k:<14}: {v}")

In [ ]:
# =====================================================================
# STEP 6: Execute Fine-Tuning on T4 GPU (Expected: ~12-15 Minutes)
# =====================================================================
print("🚀 Launching fine-tuning on RDD2022 India + Local V1...")
results = model.train(**TRAIN_CONFIG)
print("🎉 Training execution finished successfully!")

In [ ]:
# =====================================================================
# STEP 7: Checkpoint Verification & Download as 'pothole_rdd_india_best.pt'
# =====================================================================
# Explicitly renames output checkpoint to prevent overwriting pothole_best.pt
# =====================================================================
import os
import shutil
from google.colab import files

best_checkpoint = "runs/detect/pothole_rdd_india/weights/best.pt"
named_output_file = "/content/pothole_rdd_india_best.pt"

if not os.path.exists(best_checkpoint):
    raise FileNotFoundError(f"❌ ERROR: Trained checkpoint not found at {best_checkpoint}!")

# Save cleanly with explicit unique name
shutil.copy2(best_checkpoint, named_output_file)
size_mb = os.path.getsize(named_output_file) / (1024 * 1024)
print(f"✅ Trained model verified and named: {named_output_file} ({size_mb:.2f} MB)")

# 1. Save backup to Google Drive if mounted
drive_backup_dir = "/content/drive/MyDrive/RoadVision"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(drive_backup_dir, exist_ok=True)
    drive_dest = os.path.join(drive_backup_dir, "pothole_rdd_india_best.pt")
    shutil.copy2(named_output_file, drive_dest)
    print(f"✅ Backup saved to Google Drive: {drive_dest}")

# 2. Trigger browser download directly named pothole_rdd_india_best.pt
print(f"Triggering browser download for '{os.path.basename(named_output_file)}'...")
try:
    files.download(named_output_file)
    print("Download prompt sent to browser.")
except Exception as e:
    print(f"Browser download prompt error: {e}. You can download directly from the Colab file browser at '{named_output_file}'.")